# Word Embeddings From Scratch

Both classical routes to a word vector, implemented in numpy with no ML library:

1. **Count-based** — co-occurrence → PPMI → truncated SVD (this is LSA)
2. **Predictive** — skip-gram with negative sampling (word2vec's SGNS)

Then we check whether they agree. Levy & Goldberg (2014) showed SGNS is implicitly
factorizing a shifted PMI matrix, so they *should*.

**Runs offline**, about a minute. Theory: [`01_Text_Representation.md`](01_Text_Representation.md) §3.

In [ ]:
import numpy as np

RNG = np.random.default_rng(0)


## 1. A corpus with real co-occurrence structure

The distributional hypothesis only has something to find if words from the same theme
actually appear together. These sentences stay on-theme ~85% of the time, with glue words
sprinkled throughout — the glue words are the control group: they should *not* end up
near any theme.

In [ ]:
THEMES = {
    "space":    "orbit launch rocket satellite mission payload telescope lunar crew module".split(),
    "medicine": "patient clinical trial dosage symptom diagnosis treatment therapy chronic vaccine".split(),
    "finance":  "market equity portfolio dividend yield inflation trading asset liquidity valuation".split(),
}
GLUE = "the of and to in for with on a is".split()


In [ ]:
def make_sentences(n=4000, length=(8, 16), stay=0.85):
    """Words from the same theme co-occur. That structure is what embeddings learn."""
    themes = list(THEMES)
    out = []
    for _ in range(n):
        t = RNG.choice(themes)
        L = RNG.integers(*length)
        toks = []
        for _ in range(L):
            if RNG.random() < 0.25:
                toks.append(RNG.choice(GLUE))
            else:
                if RNG.random() > stay:                 # occasional theme drift
                    t = RNG.choice(themes)
                toks.append(RNG.choice(THEMES[t]))
        out.append(toks)
    return out


In [ ]:
sents = make_sentences()
print("sentences:", len(sents), "| example:", " ".join(sents[0]))


## 2. Co-occurrence counts

A symmetric window of ±4 tokens. `C[i, j]` = how often word *j* appeared near word *i*.

In [ ]:
vocab = sorted({w for s in sents for w in s})
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print("vocab:", V)


## 3. PPMI

Raw counts are dominated by frequent words, so reweight by how much more often a pair
co-occurs than chance predicts:

```
                  P(w, c)
PMI(w, c) = log  ---------
                 P(w) P(c)
```

PMI is -inf for a pair never seen together, so clamp at zero: **PPMI** = max(0, PMI).


In [ ]:
# ---------------- co-occurrence ----------------
WINDOW = 4
C = np.zeros((V, V))
for s in sents:
    ids = [w2i[w] for w in s]
    for i, wi in enumerate(ids):
        lo, hi = max(0, i - WINDOW), min(len(ids), i + WINDOW + 1)
        for j in range(lo, hi):
            if j != i:
                C[wi, ids[j]] += 1
print("co-occurrence matrix:", C.shape, "| nonzero", int((C > 0).sum()))


## 4. Truncated SVD → dense vectors

Factor the PPMI matrix and keep the top `D` components. This is **LSA** — literally PCA on a
reweighted co-occurrence matrix. Rows of the result are the word vectors.

In [ ]:
# ---------------- PPMI ----------------
total = C.sum()
Pwc = C / total
Pw = Pwc.sum(1, keepdims=True)
Pc = Pwc.sum(0, keepdims=True)
with np.errstate(divide="ignore", invalid="ignore"):
    PMI = np.log(Pwc / (Pw * Pc))
PPMI = np.nan_to_num(np.maximum(PMI, 0), nan=0.0, posinf=0.0, neginf=0.0)
print("PPMI: max %.2f, mean %.3f, %% zero %.1f" % (PPMI.max(), PPMI.mean(), 100 * (PPMI == 0).mean()))


In [ ]:
# ---------------- SVD (= LSA) ----------------
D = 24
U, S, Vt = np.linalg.svd(PPMI, full_matrices=False)
E_svd = U[:, :D] * S[:D]
E_svd /= (np.linalg.norm(E_svd, axis=1, keepdims=True) + 1e-9)
print("SVD embeddings:", E_svd.shape)


We score quality with **same-theme purity@5**: of each word's 5 nearest neighbours, what
fraction share its theme? 1.0 means the geometry perfectly recovered the themes.

In [ ]:
def neighbours(E, word, k=5):
    v = E[w2i[word]]
    sims = E @ v
    order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]


def theme_of(w):
    for t, ws in THEMES.items():
        if w in ws:
            return t
    return "glue"


def purity(E):
    hits = tot = 0
    for w in vocab:
        if theme_of(w) == "glue":
            continue
        for n, _ in neighbours(E, w, 5):
            tot += 1
            hits += theme_of(n) == theme_of(w)
    return hits / tot


## 5. Skip-gram with negative sampling

The predictive route. For each (centre, context) pair, push them together and push `k`
sampled negatives apart:

```
J = log s( v'_c . v_w )  +  SUM_{i=1..k}  E_{n ~ P_n} [ log s( -v'_n . v_w ) ]

    s  = sigmoid          v_w  = centre word vector
    k  = negatives        v'_c = context (output) vector
```

This replaces a softmax over the whole vocabulary - the thing that makes naive training
impossible - with `k+1` binary decisions.


In [ ]:
print("\n--- SVD/PPMI nearest neighbours ---")
for w in ["orbit", "vaccine", "portfolio"]:
    print(f"  {w:10s}", ", ".join(f"{n}" for n, _ in neighbours(E_svd, w)))
print(f"  same-theme purity @5: {purity(E_svd):.3f}")


Negatives come from the unigram distribution raised to the **3/4 power**, which flattens it
so rare words get sampled more often than their raw frequency would allow.

In [ ]:
# ---------------- skip-gram with negative sampling ----------------
pairs = []
for s in sents:
    ids = [w2i[w] for w in s]
    for i, wi in enumerate(ids):
        lo, hi = max(0, i - WINDOW), min(len(ids), i + WINDOW + 1)
        for j in range(lo, hi):
            if j != i:
                pairs.append((wi, ids[j]))
pairs = np.array(pairs)
print("\ntraining pairs:", len(pairs))


In [ ]:
counts = np.bincount(np.concatenate([p for p in pairs.T]), minlength=V).astype(float)
noise = counts ** 0.75
noise /= noise.sum()                      # the 3/4-power unigram distribution


### The gradient step, and one trap

`Wout` starts at zero, so every score is 0, every sigmoid is 0.5, and the initial loss is
exactly $-\log(0.5)\times(1+K)$ ≈ **4.16** for K=5. If your first epoch doesn't start near
that number, something is wrong before training even begins.

**The trap:** this vocabulary is tiny, so inside one batch the same word index recurs many
times — and `np.add.at` *sums* every occurrence. Without dividing by the batch size the
effective step is roughly `batch/vocab` times the learning rate, and training diverges.
(It did, the first time this was written.)

In [ ]:
D2, K, EPOCHS, LR = 24, 5, 5, 2.0
Win = (RNG.random((V, D2)) - 0.5) / D2
Wout = np.zeros((V, D2))


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -10, 10)))


In [ ]:
BATCH = 2048
for ep in range(EPOCHS):
    perm = RNG.permutation(len(pairs))
    total_loss, seen = 0.0, 0
    for b in range(0, len(pairs), BATCH):
        idx = perm[b:b + BATCH]
        B = len(idx)
        c, o = pairs[idx, 0], pairs[idx, 1]
        neg = RNG.choice(V, size=(len(idx), K), p=noise)

        vc = Win[c]                                    # (B, D)
        up = Wout[o]                                   # (B, D)
        un = Wout[neg]                                 # (B, K, D)

        s_pos = sigmoid(np.sum(vc * up, axis=1))
        s_neg = sigmoid(-np.einsum("bd,bkd->bk", vc, un))
        total_loss += -(np.log(s_pos + 1e-9).sum() + np.log(s_neg + 1e-9).sum())
        seen += B

        g_pos = (s_pos - 1)[:, None]                   # dL/d(score_pos)
        g_neg = (1 - s_neg)                            # (B, K)

        # Divide by B: the vocabulary is small, so one word index recurs many
        # times per batch and np.add.at SUMS every occurrence. Without the mean
        # the effective step is ~B/V times LR and training diverges.
        grad_vc = (g_pos * up + np.einsum("bk,bkd->bd", g_neg, un)) / B
        grad_up = (g_pos * vc) / B
        grad_un = (g_neg[:, :, None] * vc[:, None, :]) / B

        np.add.at(Win, c, -LR * grad_vc)
        np.add.at(Wout, o, -LR * grad_up)
        np.add.at(Wout, neg.ravel(), -LR * grad_un.reshape(-1, D2))
    print(f"  epoch {ep+1}: mean loss per pair {total_loss/seen:.4f}")


## 6. Glue words — the control

If the embeddings learned themes rather than noise, function words should cluster with each
other, not with any topic.

In [ ]:
E_sg = Win / (np.linalg.norm(Win, axis=1, keepdims=True) + 1e-9)
print("\n--- SGNS nearest neighbours ---")
for w in ["orbit", "vaccine", "portfolio"]:
    print(f"  {w:10s}", ", ".join(f"{n}" for n, _ in neighbours(E_sg, w)))
print(f"  same-theme purity @5: {purity(E_sg):.3f}")


## 7. Do the two routes agree?

In [ ]:
print("\n--- glue words should NOT be theme-pure ---")
for w in ["the", "of"]:
    print(f"  {w:10s}", ", ".join(f"{n}" for n, _ in neighbours(E_sg, w)))


## Takeaways

1. **The distributional hypothesis is the whole idea** — context defines meaning, and both
   methods are just different ways of compressing a co-occurrence matrix.
2. **PPMI before factorizing.** Raw counts encode frequency, not association.
3. **Negative sampling exists to kill the `O(|V|)` softmax**, and the 3/4-power noise
   distribution is a deliberate choice, not an implementation detail.
4. **Both routes converge on similar geometry**, which is Levy & Goldberg's result.
5. **One vector per word type.** Nothing here can represent two senses of *bank* — that is
   exactly the gap contextual embeddings fill.

**Try next:** set `stay=0.5` in `make_sentences` to blur the themes and watch purity fall;
or raise `WINDOW` and see the neighbours become more topical and less syntactic.